### Authenticate
Get the client id and secret from env vars for prod.
In dev we can create a new client each time

In [1]:
import { registerSystem } from '../v1-to-v2-data-migration/helpers/gqlHandlers.ts'
import { authenticate, getTokenForSystemClient } from '../v1-to-v2-data-migration/helpers/authentication.ts'
import { CLIENT_ID, CLIENT_SECRET } from '../v1-to-v2-data-migration/helpers/vars.ts'
import { ADMIN_USERNAME, ADMIN_PASSWORD } from '../v1-to-v2-data-migration/helpers/vars.ts'


let clientId: string | null = null
let clientSecret: string | null = null

if (CLIENT_ID && CLIENT_SECRET) {
  clientId = CLIENT_ID
  clientSecret = CLIENT_SECRET
} else {
  // For dev mode
  const adminToken = await authenticate(ADMIN_USERNAME, ADMIN_PASSWORD)
  const systemRegistration = await registerSystem(adminToken)

  clientId = systemRegistration.data.registerSystem.system.clientId
  clientSecret = systemRegistration.data.registerSystem.clientSecret
}

const sysToken = await getTokenForSystemClient(clientId, clientSecret)
sysToken

"eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJzY29wZSI6WyJyZWNvcmQuaW1wb3J0IiwicmVjb3JkLmV4cG9ydCIsInJlY29yZHNlYXJjaCIsInVzZXIuZGF0YS1zZWVkaW5nIiwicmVjb3JkLnJlaW5kZXgiLCJkZW1vIl0sInVzZXJUeXBlIjoic3lzdGVtIiwiaWF0IjoxNzY5NTIyNzQzLCJleHAiOjE3NzAxMjc1NDMsImF1ZCI6WyJvcGVuY3J2czphdXRoLXVzZXIiLCJvcGVuY3J2czp1c2VyLW1nbnQtdXNlciIsIm9wZW5jcnZzOmhlYXJ0aC11c2VyIiwib3BlbmNydnM6Z2F0ZXdheS11c2VyIiwib3BlbmNydnM6bm90aWZpY2F0aW9uLXVzZXIiLCJvcGVuY3J2czp3b3JrZmxvdy11c2VyIiwib3BlbmNydnM6c2VhcmNoLXVzZXIiLCJvcGVuY3J2czptZXRyaWNzLXVzZXIiLCJvcGVuY3J2czpjb3VudHJ5Y29uZmlnLXVzZXIiLCJvcGVuY3J2czp3ZWJob29rcy11c2VyIiwib3BlbmNydnM6Y29uZmlnLXVzZXIiLCJvcGVuY3J2czpkb2N1bWVudHMtdXNlciJdLCJpc3MiOiJvcGVuY3J2czphdXRoLXNlcnZpY2UiLCJzdWIiOiI2OTc4YzYzNzMyMzMxYjdlOGRmZWJjNjAifQ.Y6_BOWndJVqm_ekrOeyWh1_XWsxXwkzLUcraKFhh7OhP4gLTBEghXOizKQ-9mrDRk8X7GzEBID8XhNLbg2yzCRDL2C5hg2ZMYkMSkHVxFrDr5no_Ja6MtlNbMHP4OI7exIvKmsJFR283cEj7VV7DVlJ_2KX-OvzD5xZdYCRLTFr2o6jSsKpNWfPzWge3Gguu3QK-nrfofVW7_7mwmI3kNRC9SHbF-_tAq4nLEPNt99vBlp1Ww93a9hplbd4JBE0p4Oy

### Fetch data from CSVs

In [2]:
import { csvToJson } from './helpers/csvHelpers.ts'

const pathToBirthCsv = './sourceData/Birth_Register.csv'
const pathToDeathCsv = './sourceData/Death_Register.csv'
const pathToMarriageCsv = './sourceData/Marriage_Register.csv'
const pathToAdoptionCsv = './sourceData/Adoption_Register.csv'
const pathToDeedpollCsv = './sourceData/Deedpoll.csv'

const csvData = {
  birth: await csvToJson(pathToBirthCsv),
  death: await csvToJson(pathToDeathCsv),
  marriage: await csvToJson(pathToMarriageCsv),
  adoption: await csvToJson(pathToAdoptionCsv),
  deedpoll: await csvToJson(pathToDeedpollCsv),
}


In [19]:
import { GATEWAY } from "../v1-to-v2-data-migration/helpers/routes.ts";

export const getLocations = async (token: string) => {
  const response = await fetch(`${GATEWAY}/location?type=ADMIN_STRUCTURE&_count=0&status=active`, {
    method: 'GET',
    headers: {
      'Content-Type': 'application/json',
      Authorization: `Bearer ${token}`,
    },
  })
  if (!response.ok) {
    throw new Error(`Sync Locations failed: ${response.statusText}`)
  }
  return response
}

const locationsRes = await getLocations(sysToken)
const fhirLocations = await locationsRes.json()
const locationCodes = fhirLocations.entry.map((loc: any) => ({
  id: loc.resource.id,
  name: loc.resource.name,
  code: loc.resource.description
}))

locationCodes


[
  {
    id: "7e13f324-700c-4be7-8ff3-ef10ef500294",
    name: "Rarotonga",
    code: "COK-001"
  },
  {
    id: "7c2e07b6-a2f3-49df-9e70-e4e82b46dde0",
    name: "Aitutaki",
    code: "COK-002"
  },
  {
    id: "8ab75989-7777-42c0-9a5e-4c31e5e1f210",
    name: "Atiu",
    code: "COK-003"
  },
  {
    id: "a88c7c05-26a2-4e19-a10f-1c00a6ddab0f",
    name: "Mauke",
    code: "COK-004"
  },
  {
    id: "9a8afa24-5bfb-4157-b658-0ee77faf1aa9",
    name: "Mitiaro",
    code: "COK-005"
  },
  {
    id: "ee8b3512-f792-4ff1-94ac-00af70794919",
    name: "Mangaia",
    code: "COK-006"
  },
  {
    id: "bbdbacdf-27a2-41bc-9c53-d57059d9c040",
    name: "Palmerston",
    code: "COK-007"
  },
  {
    id: "5a135121-ffcf-49b0-a392-99a3f110a897",
    name: "Manihiki",
    code: "COK-008"
  },
  {
    id: "8ca687db-0e54-490d-b39c-49e669f7f959",
    name: "Rakahanga",
    code: "COK-009"
  },
  {
    id: "1e14c76b-da1b-4193-85b1-57b01a55c2f1",
    name: "Penrhyn",
    code: "COK-010"
  },
  {
    id: "a

In [ ]:
const locations = csvData.birth
  .flatMap((record) => [
    record.MOTHERS_ADDRESS,
    record.MOTHERS_BIRTHPLACE,
    record.INFORMANTS_ADDRESS,
    record.CHILDS_BIRTHPLACE,
    record.FATHERS_BIRTHPLACE,
    record.FATHERS_ADDRESS,
  ])
  .filter((x) => x)

const uniqueLocations = Array.from(new Set(locations))
  .sort()
  .map((location) => ({
    name: location,
    map: null,
  }))

Deno.writeFileSync(
  './formData/locations.json',
  new TextEncoder().encode(JSON.stringify(uniqueLocations, null, 2)),
)


### Get all potential resolvers
Use only resolvers for used event fields to avoid nulls

In [3]:
import { birthResolver } from './mappings/birthResolver.ts'
import { transform } from './helpers/transform.ts'
import { bulkImport } from '../v1-to-v2-data-migration/helpers/gqlHandlers.ts'

function nonNullObjectKeys(obj: Record<string, any>) {
  return Object.fromEntries(
    Object.entries(obj).filter(
      ([_, value]) => value !== null && value !== undefined && value !== '',
    ),
  )
}

const events = []

csvData.birth.forEach((birth) => {
  const declaration: typeof birthResolver = {}
  Object.entries(birthResolver).forEach(([eventField, dataField]) => {
    if (dataField) {
      const data = dataField(birth, csvData)
      declaration[eventField] = data
    }
  })

  const user = 'f686c526-c6b5-41ed-b3cc-43b104fa5c03'
  const location = '08260e22-bb67-4702-8760-86ec125e1079'
  const role = 'REGISTRAR'
  const event = transform(
    nonNullObjectKeys(declaration),
    'birth',
    new Date(),
    user,
    role,
    location,
    birth.BIRTH_REF,
  )
  events.push(event)
})

console.log(events.slice(0, 1))

const res = await bulkImport(events.slice(0, 1), sysToken)

JSON.stringify(res, null, 2)


[
  {
    id: "a6b5d4a5-3797-4d76-a731-12975813bdf4",
    type: "birth",
    createdAt: "2026-01-27T14:05:44.244Z",
    updatedAt: "2026-01-27T14:05:44.244Z",
    updatedAtLocation: "08260e22-bb67-4702-8760-86ec125e1079",
    trackingId: "AITU19050020",
    actions: [
      {
        type: "CREATE",
        createdAt: "2026-01-27T14:05:44.244Z",
        createdBy: "f686c526-c6b5-41ed-b3cc-43b104fa5c03",
        createdByUserType: "user",
        createdByRole: "REGISTRAR",
        createdAtLocation: "08260e22-bb67-4702-8760-86ec125e1079",
        updatedAtLocation: "08260e22-bb67-4702-8760-86ec125e1079",
        status: "Accepted",
        declaration: {},
        id: "b6e8e67e-3f9c-411d-ba95-c7b45c90d84c",
        transactionId: "81b0b1c5-f9a5-4bcc-8028-75d4efdb9f59"
      },
      {
        id: "93d50008-5363-4e95-8f3c-dfdd75c45eaa",
        type: "REGISTER",
        transactionId: "e187a560-82f1-4db6-b4b7-ce7aebc0893a",
        createdAt: "2026-01-27T14:05:44.244Z",
        createdB

"{\n" +
  '  "result": {\n' +
  '    "data": {\n' +
  '      "json": {\n' +
  '        "errors": true,\n' +
  '        "took": 0,\n' +
  '        "items": [\n' +
  "          {\n" +
  '            "index": {\n' +
  '              "_index": "events_birth",\n' +
  '              "_id": "a6b5d4a5-3797-4d76-a731-12975813bdf4",\n' +
  '              "status": 400,\n' +
  '              "error": {\n' +
  '                "type": "document_parsing_exception",\n' +
  '                "reason": "[1:1220] object mapping for [declaration.informant____address] tried to parse field [informant____address] as object, but found a concrete value"\n' +
  "              }\n" +
  "            }\n" +
  "          }\n" +
  "        ]\n" +
  "      }\n" +
  "    }\n" +
  "  }\n" +
  "}"

### Migrate births

### Migrate Deaths

In [4]:
import { reindex } from "./helpers/gqlHandlers.ts";

const reindexResponse = await reindex(sysToken);
reindexResponse


TypeError: Module not found "file:///home/baz/code/openCRVS/notebooks/cooks-migration/helpers/gqlHandlers.ts".

### Output results


In [ ]:
console.log('🍞 Declarations succesfully migrated:')
